# Task 1 — SmartRecSys: Truncated SVD

Build the User × Course interaction matrix, mean-center ratings, factorize with k=2, reconstruct predictions, and recommend 3 unseen courses.

In [ ]:
import numpy as np
import pandas as pd
from scipy.sparse.linalg import svds

df = pd.read_csv("/mnt/data/udemy_courses_with_student_ids_and_ratings.csv")
print("Dataset shape:", df.shape)
print(df.columns.tolist())
df.head()

## 1. Interaction Matrix Assembly

In [ ]:
R = df.pivot_table(
    index="student_id",
    columns="course_id",
    values="course_rating",
    aggfunc="mean"
)
print("Interaction Matrix Shape:", R.shape)
R.head()

In [ ]:
student_means = R.mean(axis=1)
print("Student baseline means:")
display(student_means.head())

In [ ]:
R_centered = R.sub(student_means, axis=0).fillna(0)
print("Mean-centered matrix:")
display(R_centered.head())

## 2. Low-Rank Factorization via SVD

$R' \approx U_k\Sigma_kV_k^T$, with $k=2$.

In [ ]:
k = 2
if k >= min(R_centered.shape):
    raise ValueError(f"k={k} is invalid for matrix shape {R_centered.shape}")

U, sigma, VT = svds(R_centered.values, k=k)

# svds returns singular values in ascending order
idx = np.argsort(sigma)[::-1]
sigma = sigma[idx]
U = U[:, idx]
VT = VT[idx, :]

print("U:", U.shape)
print("Sigma:", sigma.shape)
print("VT:", VT.shape)
print("Singular values:", sigma)

In [ ]:
R_centered_pred = U @ np.diag(sigma) @ VT
R_pred = R_centered_pred + student_means.values.reshape(-1, 1)

R_pred = pd.DataFrame(
    R_pred,
    index=R.index,
    columns=R.columns
)

display(R_pred.head())

## 3. Recommendation Function

In [ ]:
course_titles = (
    df.drop_duplicates("course_id")
      .set_index("course_id")["course_title"]
)

def recommend_top_courses(student_id, top_n=3):
    if student_id not in R.index:
        raise ValueError(f"Student ID {student_id} not found.")

    predictions = R_pred.loc[student_id].copy()

    # Remove courses already interacted with
    interacted_courses = R.loc[student_id].dropna().index
    predictions = predictions.drop(interacted_courses, errors="ignore")

    top_courses = predictions.sort_values(ascending=False).head(top_n)

    return pd.DataFrame({
        "course_id": top_courses.index,
        "course_title": [
            course_titles.get(cid, "Unknown")
            for cid in top_courses.index
        ],
        "predicted_rating": top_courses.values
    }).reset_index(drop=True)

In [ ]:
test_student_id = df["student_id"].iloc[0]
print("Test Student:", test_student_id)
display(recommend_top_courses(test_student_id, top_n=3))